# 02 - Limpeza e Tratamento

Objetivo deste notebook:
- padronizar os nomes das colunas em pt-br
- extrair `nome_municipio` e `sigla_uf`
- criar `ano = 2010`
- normalizar nomes de municipio para facilitar cruzamentos
- criar `codigo_uf_ibge`
- preparar a base para receber `codigo_municipio` de 7 digitos via tabela auxiliar


## 1. Imports e configuracao

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

PASTA_IDH = Path('../datasets/idh')
ARQUIVO_IDH = PASTA_IDH / 'data.csv'
ANO_REFERENCIA = 2010
CAMINHO_TABELA_MUNICIPIOS = Path('../datasets/auxiliares/codigos_municipios_ibge.csv')

## 2. Leitura do arquivo bruto

In [ ]:
df_idh_raw = pd.read_excel(ARQUIVO_IDH)
df_idh_raw.head()

## 3. Renomear colunas para o padrao do projeto

In [ ]:
mapa_colunas = {
    'Territorialidade': 'territorialidade',
    'Posição IDHM': 'posicao_idhm',
    'IDHM': 'idhm',
    'Posição IDHM Renda': 'posicao_idhm_renda',
    'IDHM Renda': 'idhm_renda',
    'Posição IDHM Educação': 'posicao_idhm_educacao',
    'IDHM Educação': 'idhm_educacao',
    'Posição IDHM Longevidade': 'posicao_idhm_longevidade',
    'IDHM Longevidade': 'idhm_longevidade',
}

df_idh = df_idh_raw.rename(columns=mapa_colunas).copy()
df_idh.columns.tolist()

## 4. Criar coluna de ano e extrair municipio e UF

In [ ]:
df_idh['ano'] = ANO_REFERENCIA
df_idh['sigla_uf'] = df_idh['territorialidade'].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0]
df_idh['nome_municipio'] = df_idh['territorialidade'].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True)

display(df_idh[['territorialidade', 'nome_municipio', 'sigla_uf']].head(10))

## 5. Padronizacao textual

Vamos manter o nome original do municipio e criar uma versao padronizada para cruzamentos:
- sem acentos
- em minusculas
- sem espacos excedentes


In [ ]:
def normalizar_texto(texto: str) -> str:
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(caractere for caractere in texto if not unicodedata.combining(caractere))
    texto = re.sub(r'\s+', ' ', texto)
    return texto

df_idh['nome_municipio_padronizado'] = df_idh['nome_municipio'].apply(normalizar_texto)
df_idh['sigla_uf'] = df_idh['sigla_uf'].astype(str).str.strip().str.upper()

display(df_idh[['nome_municipio', 'nome_municipio_padronizado', 'sigla_uf']].head(10))

## 6. Criar codigo da UF no padrao IBGE

Observacao importante:
- `codigo_uf_ibge` tem 2 digitos
- `codigo_municipio` e o codigo de 7 digitos e sera adicionado por cruzamento com uma tabela auxiliar


In [ ]:
mapa_codigo_uf_ibge = {
    'RO': 11,
    'AC': 12,
    'AM': 13,
    'RR': 14,
    'PA': 15,
    'AP': 16,
    'TO': 17,
    'MA': 21,
    'PI': 22,
    'CE': 23,
    'RN': 24,
    'PB': 25,
    'PE': 26,
    'AL': 27,
    'SE': 28,
    'BA': 29,
    'MG': 31,
    'ES': 32,
    'RJ': 33,
    'SP': 35,
    'PR': 41,
    'SC': 42,
    'RS': 43,
    'MS': 50,
    'MT': 51,
    'GO': 52,
    'DF': 53,
}

df_idh['codigo_uf_ibge'] = df_idh['sigla_uf'].map(mapa_codigo_uf_ibge)
df_idh['codigo_municipio'] = pd.NA

display(df_idh[['nome_municipio', 'sigla_uf', 'codigo_uf_ibge', 'codigo_municipio']].head(10))

## 7. Ajustar tipos numericos

In [ ]:
colunas_numericas = [
    'posicao_idhm',
    'idhm',
    'posicao_idhm_renda',
    'idhm_renda',
    'posicao_idhm_educacao',
    'idhm_educacao',
    'posicao_idhm_longevidade',
    'idhm_longevidade',
    'ano',
    'codigo_uf_ibge',
]

for coluna in colunas_numericas:
    df_idh[coluna] = pd.to_numeric(df_idh[coluna], errors='coerce')

df_idh[colunas_numericas].dtypes

## 8. Reordenar colunas principais

In [ ]:
colunas_ordenadas = [
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'posicao_idhm',
    'posicao_idhm_renda',
    'posicao_idhm_educacao',
    'posicao_idhm_longevidade',
    'territorialidade',
]

df_idh = df_idh[colunas_ordenadas].copy()
display(df_idh.head())

## 9. Validacoes finais da base tratada

In [ ]:
print('Linhas:', df_idh.shape[0])
print('Colunas:', df_idh.shape[1])
print('Duplicatas exatas:', df_idh.duplicated().sum())
print('Duplicatas em nome_municipio + sigla_uf + ano:', df_idh.duplicated(subset=['nome_municipio', 'sigla_uf', 'ano']).sum())
print('Nulos por coluna:')
display(df_idh.isna().sum().to_frame('qtd_nulos'))

## 10. Cruzamento opcional para adicionar codigo_municipio

Quando a tabela auxiliar estiver disponivel, o merge recomendado e por:
- `sigla_uf`
- `nome_municipio_padronizado`

Estrutura esperada da tabela auxiliar:
- `codigo_municipio`
- `codigo_uf_ibge`
- `sigla_uf`
- `nome_municipio`
- `nome_municipio_padronizado`


In [ ]:
if CAMINHO_TABELA_MUNICIPIOS.exists():
    df_codigos = pd.read_csv(CAMINHO_TABELA_MUNICIPIOS)
    df_codigos['nome_municipio_padronizado'] = df_codigos['nome_municipio_padronizado'].apply(normalizar_texto)
    df_codigos['sigla_uf'] = df_codigos['sigla_uf'].astype(str).str.upper().str.strip()

    df_idh = df_idh.drop(columns=['codigo_municipio']).merge(
        df_codigos[['codigo_municipio', 'sigla_uf', 'nome_municipio_padronizado']],
        how='left',
        on=['sigla_uf', 'nome_municipio_padronizado'],
        validate='one_to_one'
    )

    display(df_idh.head())
    print('Municipios sem codigo_municipio:', df_idh['codigo_municipio'].isna().sum())
else:
    print(f'Tabela auxiliar nao encontrada em: {CAMINHO_TABELA_MUNICIPIOS}')
    print('A base tratada foi preparada para receber o codigo_municipio em uma proxima etapa.')

## 11. Resultado da etapa

Ao final deste notebook, a base de IDH deve estar:
- lida corretamente
- com colunas em pt-br
- com `ano = 2010`
- com `nome_municipio` e `sigla_uf` extraidos
- com `nome_municipio_padronizado` criado
- com `codigo_uf_ibge` preenchido
- pronta para receber `codigo_municipio` e ser integrada aos demais datasets
